# Семинар №10. PL/pgSQL, триггеры, VIEW и MATERIALIZED VIEW

## Как устроен ноутбук

1. Каждая тема идёт по схеме: **теория → демо → упражнение**.
2. Ячейки с комментарием `# УПРАЖНЕНИЕ N` — для самостоятельной работы; там стоят `TODO`.
3. Полные решения собраны в разделе «Решения» в самом конце. Старайтесь сначала решить сами.
4. Выполняйте ячейки **по порядку**: состояние таблиц в БД важно (мы переиспользуем одни и те же таблицы).
5. Сброс: `Kernel → Restart & Run All`.

## Окружение

- PostgreSQL 16 слушает `localhost:5432`.
- Учётные данные БД: пользователь `postgres`, пароль не требуется (trust auth), база `demo`.

## План

1. Функции (`IMMUTABLE` / `STABLE` / `VOLATILE`) и процедуры с `COMMIT`.
2. Динамический SQL через `format()` и защита от инъекций.
3. Обработка ошибок: `EXCEPTION`, `GET STACKED DIAGNOSTICS`.
4. AFTER-триггеры: `FOR EACH ROW` и `FOR EACH STATEMENT`.
5. BEFORE-триггеры: модификация полей и запрет операций.
6. VIEW, `WITH CHECK OPTION`, `INSTEAD OF`-триггеры.
7. MATERIALIZED VIEW и `REFRESH CONCURRENTLY`.


In [1]:
import sys
print("Python:", sys.version.split()[0])

import psycopg
print("psycopg:", psycopg.__version__)

DB = {"host": "localhost", "port": 5432, "dbname": "demo", "user": "postgres"}
PSYCOPG_CONN = (
    f"host={DB['host']} port={DB['port']} dbname={DB['dbname']} user={DB['user']}"
)
print("DSN    :", PSYCOPG_CONN)


def q(sql, params=None):
    """Выполнить SQL и вернуть строки (для SELECT) либо None.

    Соединение открывается на один вызов в режиме autocommit — это удобно
    для DDL и процедур с COMMIT, которые мы демонстрируем ниже.
    """
    with psycopg.connect(PSYCOPG_CONN, autocommit=True) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall() if cur.description else None

Python: 3.11.17
psycopg: 3.3.6
DSN    : host=localhost port=5432 dbname=demo user=postgres


In [2]:
# --- Подготовка учебных таблиц ---
q("DROP TABLE IF EXISTS ticket_flights, flights, iot_telemetry, "
  "loan_applications, audit_log, equipment, error_log CASCADE")

q("""
CREATE TABLE flights (
    flight_id SERIAL PRIMARY KEY,
    flight_no VARCHAR(10),
    departure_airport VARCHAR(3),
    scheduled_departure TIMESTAMP
)
""")
q("""
INSERT INTO flights (flight_no, departure_airport, scheduled_departure) VALUES
  ('SU1234', 'DME', '2025-06-15 08:00:00'),
  ('SU5678', 'DME', '2025-06-15 12:30:00'),
  ('AB999',  'SVO', '2025-06-15 14:00:00')
""")

q("""
CREATE TABLE ticket_flights (
    ticket_no VARCHAR(13),
    flight_id INT REFERENCES flights(flight_id),
    amount NUMERIC
)
""")
q("""
INSERT INTO ticket_flights VALUES
  ('1234567890', 1, 15000.00),
  ('1234567891', 1, 18000.50),
  ('1234567892', 2, 22000.00)
""")

q("""
CREATE TABLE iot_telemetry (
    id SERIAL PRIMARY KEY,
    sensor_id VARCHAR(20),
    temperature NUMERIC,
    reading_ts TIMESTAMP
)
""")

q("""
CREATE TABLE loan_applications (
    id SERIAL PRIMARY KEY,
    borrower_name TEXT,
    amount NUMERIC,
    status VARCHAR(20) DEFAULT 'NEW',
    updated_at TIMESTAMP
)
""")
q("""
CREATE TABLE audit_log (
    log_id SERIAL PRIMARY KEY,
    table_name TEXT,
    operation TEXT,
    changed_by TEXT DEFAULT current_user,
    changed_at TIMESTAMP DEFAULT now(),
    old_data JSONB,
    new_data JSONB
)
""")
q("""
CREATE TABLE equipment (
    id SERIAL PRIMARY KEY,
    name TEXT,
    status VARCHAR(20) DEFAULT 'ACTIVE',
    updated_at TIMESTAMP
)
""")
q("""
CREATE TABLE error_log (
    id SERIAL PRIMARY KEY,
    proc_name TEXT,
    error_msg TEXT,
    error_detail TEXT,
    logged_at TIMESTAMP DEFAULT now()
)
""")

print("Таблицы созданы. flights:", q("SELECT count(*) FROM flights")[0][0])

Таблицы созданы. flights: 3


## 1. Функции и процедуры

**Функция** (`CREATE FUNCTION`) возвращает значение и вызывается в `SELECT`, `FROM` или выражениях. **Процедура** (`CREATE PROCEDURE`) вызывается через `CALL`, может управлять транзакциями (`COMMIT`/`ROLLBACK`) — но только если `CALL` выполняется вне явного блока транзакции (в psycopg с `autocommit=True` это так).

**Категории волатильности** — обещание планировщику:
- `IMMUTABLE` — результат зависит только от аргументов (чистая функция). Допускается в выражениях индексов.
- `STABLE` — зависит от состояния БД (например, читает таблицу), но не меняет её. В индексах запрещена.
- `VOLATILE` (по умолчанию) — может что угодно, не кэшируется.

**Правило:** всегда указывайте категорию явно.

In [3]:
# Демо 1.1: скалярная функция с IMMUTABLE
q("""
CREATE OR REPLACE FUNCTION calculate_tax(amount NUMERIC, tax_rate NUMERIC DEFAULT 0.13)
RETURNS NUMERIC
AS $$
BEGIN
    IF amount < 0 THEN
        RAISE EXCEPTION 'Сумма не может быть отрицательной: %', amount;
    END IF;
    RETURN amount * tax_rate;
END;
$$ LANGUAGE plpgsql IMMUTABLE
""")

for row in q("SELECT flight_id, amount, calculate_tax(amount) FROM ticket_flights ORDER BY flight_id"):
    print(row)

(1, Decimal('15000.00'), Decimal('1950.0000'))
(1, Decimal('18000.50'), Decimal('2340.0650'))
(2, Decimal('22000.00'), Decimal('2860.0000'))


In [4]:
# Демо 1.2: табличная функция со STABLE
q("""
CREATE OR REPLACE FUNCTION get_flights_by_airport(p_airport_code VARCHAR)
RETURNS TABLE (flight_id INT, flight_no VARCHAR, scheduled_departure TIMESTAMP) AS $$
BEGIN
    RETURN QUERY
    SELECT f.flight_id, f.flight_no, f.scheduled_departure
    FROM flights f
    WHERE f.departure_airport = p_airport_code
    ORDER BY f.scheduled_departure DESC;
END;
$$ LANGUAGE plpgsql STABLE
""")

for row in q("SELECT * FROM get_flights_by_airport('DME')"):
    print(row)

(2, 'SU5678', datetime.datetime(2025, 6, 15, 12, 30))
(1, 'SU1234', datetime.datetime(2025, 6, 15, 8, 0))


In [5]:
# Демо 1.3: динамический SQL через format()
# %I — идентификатор (имя таблицы, колонки), %L — литерал (значение).
# Никаких конкатенаций строк с пользовательским вводом.
q("""
CREATE OR REPLACE FUNCTION count_rows(tbl_name text)
RETURNS bigint AS $$
DECLARE
    result bigint;
BEGIN
    EXECUTE format('SELECT count(*) FROM %I', tbl_name) INTO result;
    RETURN result;
END;
$$ LANGUAGE plpgsql STABLE
""")

print("count_rows('flights')       :", q("SELECT count_rows('flights')")[0][0])
print("count_rows('ticket_flights'):", q("SELECT count_rows('ticket_flights')")[0][0])

count_rows('flights')       : 3
count_rows('ticket_flights'): 3


In [6]:
# Демо 1.4: процедура с COMMIT
# COMMIT каждые 1000 строк — можно прервать и не потерять уже записанное.
# Работает только в autocommit-режиме (наш q() его использует).
q("""
CREATE OR REPLACE PROCEDURE bulk_insert_logs(p_count INT)
AS $$
DECLARE
    i INT;
BEGIN
    FOR i IN 1..p_count LOOP
        INSERT INTO iot_telemetry (sensor_id, temperature, reading_ts)
        VALUES ('PROC_TEST', random()*100, now());
        IF i % 1000 = 0 THEN
            COMMIT;
            RAISE NOTICE 'Закоммичено % строк', i;
        END IF;
    END LOOP;
    COMMIT;
END;
$$ LANGUAGE plpgsql
""")

q("CALL bulk_insert_logs(3000)")
print("Всего строк PROC_TEST:",
      q("SELECT count(*) FROM iot_telemetry WHERE sensor_id = 'PROC_TEST'")[0][0])

Всего строк PROC_TEST: 3000


In [7]:
# Демо 1.5: обработка ошибок и GET STACKED DIAGNOSTICS
q("""
CREATE OR REPLACE FUNCTION safe_divide(a NUMERIC, b NUMERIC)
RETURNS NUMERIC AS $$
BEGIN
    RAISE NOTICE 'Делим % на %', a, b;
    IF b = 0 THEN
        RAISE EXCEPTION 'Деление на ноль';
    END IF;
    RETURN a / b;
EXCEPTION WHEN OTHERS THEN
    DECLARE
        err_detail TEXT;
    BEGIN
        GET STACKED DIAGNOSTICS err_detail = PG_EXCEPTION_DETAIL;
        RAISE WARNING 'Ошибка: %, детали: %', SQLERRM, err_detail;
        RETURN NULL;
    END;
END;
$$ LANGUAGE plpgsql STABLE
""")

print("safe_divide(10, 4) =", q("SELECT safe_divide(10, 4)")[0][0])
print("safe_divide(10, 0) =", q("SELECT safe_divide(10, 0)")[0][0])

safe_divide(10, 4) = 2.5000000000000000
safe_divide(10, 0) = None


### Упражнения по разделу 1

**Упражнение 1. `flight_label`**

Напишите скалярную функцию `flight_label(p_flight_no VARCHAR, p_airport VARCHAR) RETURNS TEXT`, возвращающую строку вида `'SU1234 (DME)'`. Пометьте корректной категорией волатильности. Выведите метки всех рейсов:

```sql
SELECT flight_label(flight_no, departure_airport) FROM flights;
```

Ожидаемый вывод: `SU1234 (DME)`, `SU5678 (DME)`, `AB999 (SVO)`.

**Упражнение 2. `tickets_for_flight`**

Напишите табличную функцию `tickets_for_flight(p_flight_id INT) RETURNS TABLE(ticket_no VARCHAR, amount NUMERIC)`, возвращающую все билеты на указанный рейс, отсортированные по `amount` по убыванию. Категория — `STABLE`. Проверьте вызов `SELECT * FROM tickets_for_flight(1);` — должны увидеть два билета.

In [ ]:
# УПРАЖНЕНИЕ 1

# TODO: CREATE OR REPLACE FUNCTION flight_label(p_flight_no VARCHAR, p_airport VARCHAR)
# TODO: RETURNS TEXT ... (внутри RETURN p_flight_no || ' (' || p_airport || ')';)
# TODO: LANGUAGE plpgsql + правильная категория волатильности

# TODO: проверка:
# for row in q("SELECT flight_label(flight_no, departure_airport) FROM flights"):
#     print(row)

In [ ]:
# УПРАЖНЕНИЕ 2

# TODO: CREATE OR REPLACE FUNCTION tickets_for_flight(p_flight_id INT)
# TODO: RETURNS TABLE(ticket_no VARCHAR, amount NUMERIC) AS $$ ... RETURN QUERY ... $$ STABLE

# TODO: for row in q("SELECT * FROM tickets_for_flight(1)"):
#     print(row)

## 2. AFTER-триггеры: ROW и STATEMENT

**`AFTER`** срабатывает **после** записи строки — идеален для аудита и уведомлений. Возврат значения игнорируется (можно возвращать `NULL`).

**`FOR EACH ROW`** — вызывается на каждую изменённую строку, даёт доступ к `NEW` и `OLD`.
**`FOR EACH STATEMENT`** — вызывается один раз на SQL-оператор. `NEW`/`OLD` недоступны. Дешевле, но без деталей.

**Порядок срабатывания** триггеров одного типа — алфавитный по имени. Не полагайтесь на это в сложной логике.

In [ ]:
# Демо 2.1: ROW-триггер для детального аудита
q("""
CREATE OR REPLACE FUNCTION log_application_changes_row()
RETURNS TRIGGER AS $$
BEGIN
    IF (TG_OP = 'DELETE') THEN
        INSERT INTO audit_log (table_name, operation, old_data)
        VALUES (TG_TABLE_NAME, 'D', to_jsonb(OLD));
    ELSIF (TG_OP = 'UPDATE') THEN
        INSERT INTO audit_log (table_name, operation, old_data, new_data)
        VALUES (TG_TABLE_NAME, 'U', to_jsonb(OLD), to_jsonb(NEW));
    ELSIF (TG_OP = 'INSERT') THEN
        INSERT INTO audit_log (table_name, operation, new_data)
        VALUES (TG_TABLE_NAME, 'I', to_jsonb(NEW));
    END IF;
    RETURN NULL;
END;
$$ LANGUAGE plpgsql
""")

q("""
CREATE TRIGGER trg_audit_loan_row
AFTER INSERT OR UPDATE OR DELETE ON loan_applications
FOR EACH ROW EXECUTE FUNCTION log_application_changes_row()
""")

q("INSERT INTO loan_applications (borrower_name, amount) VALUES ('Анна', 100000), ('Борис', 200000)")

for row in q("SELECT table_name, operation, new_data FROM audit_log"):
    print(row)

In [ ]:
# Демо 2.2: STATEMENT-триггер для лёгкого логирования
q("""
CREATE OR REPLACE FUNCTION log_operation_statement()
RETURNS TRIGGER AS $$
BEGIN
    INSERT INTO audit_log (table_name, operation, changed_by)
    VALUES (TG_TABLE_NAME, TG_OP, current_user);
    RETURN NULL;
END;
$$ LANGUAGE plpgsql
""")

q("""
CREATE TRIGGER trg_audit_loan_stmt
AFTER INSERT OR UPDATE OR DELETE ON loan_applications
FOR EACH STATEMENT EXECUTE FUNCTION log_operation_statement()
""")

# Один оператор INSERT на две строки → 2 записи ROW + 1 запись STATEMENT
q("INSERT INTO loan_applications (borrower_name, amount) VALUES ('Виктор', 50000), ('Галина', 75000)")

for row in q("SELECT table_name, operation, new_data IS NOT NULL AS has_details FROM audit_log ORDER BY log_id"):
    print(row)

### Упражнение по разделу 2

**Упражнение 3. Аудит для `equipment`**

Создайте ROW-триггер аудита для таблицы `equipment` (INSERT/UPDATE/DELETE → запись в `audit_log`). Используйте уже созданную функцию `log_application_changes_row()` — она универсальна и берёт имя таблицы из `TG_TABLE_NAME`.

Затем:
1. Вставьте две записи в `equipment`.
2. Обновите одну из них.
3. Удалите одну.
4. Выведите `operation` из `audit_log` для `table_name = 'equipment'`. Ожидаемая последовательность: `I`, `I`, `U`, `D`.

In [ ]:
# УПРАЖНЕНИЕ 3

# TODO: CREATE TRIGGER trg_audit_equipment_row AFTER INSERT OR UPDATE OR DELETE
# TODO:   ON equipment FOR EACH ROW EXECUTE FUNCTION log_application_changes_row();

# TODO: вставьте 2 записи, обновите одну, удалите одну

# TODO: SELECT operation FROM audit_log WHERE table_name = 'equipment'
# for row in q("SELECT operation FROM audit_log WHERE table_name = 'equipment' ORDER BY log_id"):
#     print(row[0])

## 3. BEFORE-триггеры: модификация и запрет операций

**`BEFORE`** срабатывает **до** записи строки. Можно:
- изменить `NEW` (автозаполнение полей);
- отменить операцию через `RAISE EXCEPTION`;
- тихо отменить, вернув `NULL` (без ошибки).

Подходит для инфраструктурных инвариантов: `updated_at`, жёсткие запреты на удаление активных сущностей. Не подходит для бизнес-правил (расчёт ставок, проверка заявок) — это место в прикладном коде.

In [ ]:
# Демо 3.1: BEFORE-триггер с двумя ветками
q("""
CREATE OR REPLACE FUNCTION equipment_maintenance()
RETURNS TRIGGER AS $$
BEGIN
    IF (TG_OP = 'UPDATE') THEN
        NEW.updated_at := now();
    ELSIF (TG_OP = 'DELETE') THEN
        IF OLD.status = 'ACTIVE' THEN
            RAISE EXCEPTION 'Нельзя удалить активное оборудование (id=%)', OLD.id;
        END IF;
        RETURN OLD;
    END IF;
    RETURN NEW;
END;
$$ LANGUAGE plpgsql
""")

q("""
CREATE TRIGGER trg_equipment_before
BEFORE UPDATE OR DELETE ON equipment
FOR EACH ROW EXECUTE FUNCTION equipment_maintenance()
""")

q("INSERT INTO equipment (name) VALUES ('Сервер A')")
q("UPDATE equipment SET name = 'Сервер B' WHERE id = 1")
print("после UPDATE:", q("SELECT id, name, updated_at FROM equipment"))

# Попытка удалить активное оборудование:
try:
    q("DELETE FROM equipment WHERE id = 1")
except psycopg.errors.RaiseException as e:
    print("DELETE ACTIVE →", str(e).splitlines()[0])

q("UPDATE equipment SET status = 'INACTIVE' WHERE id = 1")
q("DELETE FROM equipment WHERE id = 1")
print("после DELETE:", q("SELECT count(*) FROM equipment")[0][0], "строк(и)")

### Упражнение по разделу 3

**Упражнение 4. `updated_at` для `loan_applications`**

Создайте `BEFORE UPDATE`-триггер на таблицу `loan_applications`, который:
1. Автоматически проставляет `NEW.updated_at := now()`.
2. Запрещает через `RAISE EXCEPTION` изменение `status` у заявок со статусом `'APPROVED'`.

Проверка:
1. Вставьте заявку со `status = 'APPROVED'`.
2. Попробуйте `UPDATE loan_applications SET status = 'REJECTED' WHERE id = ...` — должно упасть.
3. `UPDATE loan_applications SET amount = 999 WHERE id = ...` — должно пройти, и `updated_at` заполниться.

In [ ]:
# УПРАЖНЕНИЕ 4

# TODO: CREATE OR REPLACE FUNCTION loan_applications_before()
# TODO: в UPDATE-ветке: NEW.updated_at := now();
# TODO: если OLD.status = 'APPROVED' AND NEW.status <> OLD.status -> RAISE EXCEPTION

# TODO: CREATE TRIGGER trg_loan_before BEFORE UPDATE ON loan_applications
# TODO:   FOR EACH ROW EXECUTE FUNCTION loan_applications_before();

# TODO: проверки (вставка APPROVED, попытка смены статуса, смена amount)

## 4. VIEW и MATERIALIZED VIEW

**`VIEW`** — виртуальная таблица: SQL-запрос не выполняется при создании и не хранится в БД. Данные всегда актуальны, читаются из базовых таблиц в момент обращения.

**`MATERIALIZED VIEW`** — результат запроса физически хранится как таблица. Чтение мгновенное, можно строить индексы. Но данные **устаревают** и требуют `REFRESH`.

**Обновляемость VIEW:**
- простое VIEW над одной таблицей без агрегации — обновляемо автоматически;
- `WITH CHECK OPTION` запрещает вставлять/менять строки, не удовлетворяющие `WHERE` представления;
- сложное VIEW (с `GROUP BY`, `JOIN`) — необновляемо; для записи через него нужен `INSTEAD OF`-триггер.

**`REFRESH MATERIALIZED VIEW CONCURRENTLY`** позволяет читать старую версию во время пересчёта, но требует **уникальный индекс** на MV без `WHERE`.

In [ ]:
# Демо 4.1: обновляемое VIEW с WITH CHECK OPTION
q("""
CREATE OR REPLACE VIEW v_dme_flights AS
SELECT flight_id, flight_no, departure_airport, scheduled_departure
FROM flights
WHERE departure_airport = 'DME'
WITH CHECK OPTION
""")

# Вставка через VIEW с соблюдением условия — работает
q("""INSERT INTO v_dme_flights (flight_no, departure_airport, scheduled_departure)
     VALUES ('SU9999', 'DME', '2025-06-15 10:00:00')""")
print("после INSERT DME:", q("SELECT * FROM flights WHERE flight_no = 'SU9999'"))

# Вставка с нарушением условия — отклоняется WITH CHECK OPTION
try:
    q("""INSERT INTO v_dme_flights (flight_no, departure_airport, scheduled_departure)
         VALUES ('SU0001', 'SVO', '2025-06-15 11:00:00')""")
except psycopg.errors.Error as e:
    print("INSERT SVO через v_dme_flights →", str(e).splitlines()[0])

In [ ]:
# Демо 4.2: MATERIALIZED VIEW + REFRESH CONCURRENTLY
q("""
CREATE MATERIALIZED VIEW mv_flight_revenue AS
SELECT f.flight_id, f.flight_no,
       COUNT(tf.ticket_no) AS tickets,
       COALESCE(SUM(tf.amount), 0) AS revenue
FROM flights f
LEFT JOIN ticket_flights tf ON f.flight_id = tf.flight_id
GROUP BY f.flight_id, f.flight_no
""")

# Уникальный индекс — обязательное условие для REFRESH CONCURRENTLY
q("CREATE UNIQUE INDEX idx_mv_flight_revenue_pk ON mv_flight_revenue(flight_id)")

print("MV сразу после создания:")
for row in q("SELECT * FROM mv_flight_revenue ORDER BY flight_id"):
    print(" ", row)

# Добавим билет — но MV не обновится автоматически
q("INSERT INTO ticket_flights VALUES ('1234567893', 1, 25000.00)")
print("\nMV ДО REFRESH (flight_id=1):", q("SELECT tickets, revenue FROM mv_flight_revenue WHERE flight_id = 1"))

q("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_flight_revenue")
print("MV ПОСЛЕ REFRESH (flight_id=1):", q("SELECT tickets, revenue FROM mv_flight_revenue WHERE flight_id = 1"))

In [ ]:
# Демо 4.3: INSTEAD OF-триггер на необновляемом VIEW (с GROUP BY)
q("""
CREATE OR REPLACE VIEW v_flight_revenue AS
SELECT f.flight_id, f.flight_no, COALESCE(SUM(tf.amount), 0) AS revenue
FROM flights f
LEFT JOIN ticket_flights tf ON f.flight_id = tf.flight_id
GROUP BY f.flight_id, f.flight_no
""")

q("""
CREATE OR REPLACE FUNCTION trg_update_flight_revenue()
RETURNS TRIGGER AS $$
BEGIN
    IF TG_OP = 'UPDATE' THEN
        UPDATE flights SET flight_no = NEW.flight_no WHERE flight_id = NEW.flight_id;
        RETURN NEW;
    ELSIF TG_OP = 'DELETE' THEN
        DELETE FROM flights WHERE flight_id = OLD.flight_id;
        RETURN OLD;
    END IF;
    RETURN NULL;
END;
$$ LANGUAGE plpgsql
""")

q("""
CREATE TRIGGER trg_v_flight_revenue
INSTEAD OF UPDATE OR DELETE ON v_flight_revenue
FOR EACH ROW EXECUTE FUNCTION trg_update_flight_revenue()
""")

q("UPDATE v_flight_revenue SET flight_no = 'SU1234NEW' WHERE flight_id = 1")
print("flights после UPDATE через VIEW:", q("SELECT flight_id, flight_no FROM flights WHERE flight_id = 1"))

### Упражнения по разделу 4

**Упражнение 5. Обновляемое VIEW с `WITH CHECK OPTION`**

Создайте VIEW `v_svo_flights` с полями `flight_id, flight_no, departure_airport, scheduled_departure` и фильтром `departure_airport = 'SVO'`, добавьте `WITH CHECK OPTION`.

Проверьте:
1. `INSERT INTO v_svo_flights (...) VALUES ('XX111', 'SVO', ...)` — работает.
2. `INSERT INTO v_svo_flights (...) VALUES ('XX222', 'DME', ...)` — отклоняется.
3. `SELECT count(*) FROM v_svo_flights` — отражает только SVO-рейсы.

**Упражнение 6. MV по аэропортам + `REFRESH CONCURRENTLY`**

Создайте MATERIALIZED VIEW `mv_airport_stats`, агрегирующий количество рейсов по `departure_airport`:

```sql
SELECT departure_airport, count(*) AS flights_cnt
FROM flights
GROUP BY departure_airport;
```

1. Создайте уникальный индекс по `departure_airport`.
2. Выведите содержимое MV.
3. Добавьте новый рейс в `flights`.
4. Убедитесь, что MV не изменился.
5. Выполните `REFRESH MATERIALIZED VIEW CONCURRENTLY` и убедитесь, что данные обновились.

In [ ]:
# УПРАЖНЕНИЕ 5

# TODO: CREATE OR REPLACE VIEW v_svo_flights AS SELECT ... WHERE departure_airport='SVO' WITH CHECK OPTION;
# TODO: INSERT корректный SVO — должен пройти
# TODO: INSERT SVO/DME — должен упасть (try/except)
# TODO: SELECT count(*) FROM v_svo_flights

In [ ]:
# УПРАЖНЕНИЕ 6

# TODO: CREATE MATERIALIZED VIEW mv_airport_stats AS SELECT departure_airport, count(*) AS flights_cnt FROM flights GROUP BY departure_airport;
# TODO: CREATE UNIQUE INDEX idx_mv_airport_stats ON mv_airport_stats(departure_airport);
# TODO: вывести MV
# TODO: INSERT INTO flights (...) VALUES ('NEW01', 'LED', ...);
# TODO: снова вывести MV (должно быть старое значение)
# TODO: REFRESH MATERIALIZED VIEW CONCURRENTLY mv_airport_stats;
# TODO: вывести MV (появился LED)

## Итоги семинара

1. **Функции vs процедуры:** функции возвращают значение и участвуют в запросах; процедуры вызываются через `CALL` и могут управлять транзакциями (только вне явных блоков).
2. **Волатильность:** `IMMUTABLE` для чистых вычислений и индексов, `STABLE` для чтения таблиц, `VOLATILE` по умолчанию. Указывайте явно.
3. **Динамический SQL:** только через `format()` с `%I` (идентификатор) и `%L` (литерал). Никакой конкатенации.
4. **Обработка ошибок:** `EXCEPTION` — неявный savepoint (дорогой). `GET STACKED DIAGNOSTICS` даёт детали. В бою — `RAISE;` после логирования.
5. **Триггеры AFTER** — аудит и уведомления. `FOR EACH ROW` даёт `NEW`/`OLD`, `FOR EACH STATEMENT` дешевле, но без деталей.
6. **Триггеры BEFORE** — автозаполнение полей и жёсткие инварианты. `RETURN NEW` пропускает, `RAISE EXCEPTION` отменяет, `RETURN NULL` тихо отменяет строку.
7. **VIEW:** простое — обновляемо, `WITH CHECK OPTION` фильтрует записи; сложное — через `INSTEAD OF`-триггер.
8. **MATERIALIZED VIEW:** физически хранится, требует `REFRESH`. Для `CONCURRENTLY` нужен уникальный индекс. Актуальность — на совести расписания (cron/pg_cron).
9. **В микросервисах:** триггеры — только для инфраструктуры (аудит, инварианты состояния). Бизнес-логика — в сервисах.